<a href="https://colab.research.google.com/github/tekpinar/gromacscolab/blob/main/MD_TrajectoryAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**GROMACSCOLAB: TRAJECTORY ANALYSES NOTEBOOK**

**Molecular Dynamics** (**MD**) simulation is a powerful computational method used to understand the atomic-level behavior of biomolecules and to study the thermodynamic and kinetic properties of systems. **GROMACS**, with its high-performance computing capabilities and flexible user interface, is widely preferred for performing MD simulations. In this notebook, we will analyze a molecular dynamics simulation performed with GROMACS. **You will need to upload a trajectory containing only the protein to conduct this analysis.**





In [ ]:
#@title Install GROMACS and test it by issuing `gmx` command
!apt install gromacs >/dev/null
!gmx

In [ ]:
#@title Install the required Python libraries
!pip install py3Dmol
!pip install nglview
!pip install mdanalysis biopython

In [ ]:
#@title Upload your PDB and trajectory file containing only protein! {run: "auto"}
from google.colab import files

print("Upload your PDB file:")
uploaded_pdb = files.upload()
pdb_filename = list(uploaded_pdb.keys())[0]

print("\nUpload your trajectory file (xtc, trr):")
uploaded_traj = files.upload()
traj_filename = list(uploaded_traj.keys())[0]

print(f"\nPDB file uploaded: {pdb_filename}")
print(f"Trajectory file uploaded: {traj_filename}")

This command enables **interactive widget support** in Google Colab, allowing **real-time visualization and analysis** of GROMACS simulation data.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
# import MDAnalysis as mda
# import nglview as nv
# from nglview.datafiles import PDB, XTC

# u = mda.Universe(pdb_filename, traj_filename)

# protein = u.select_atoms('protein')

# w = nv.show_mdanalysis(protein)
# w

# Root Mean Square Deviation (RMSD)
### Description:
RMSD measures the structural deviations of a molecule over time. RMSD reflects the overall change of the entire structure relative to its initial conformation. RMSD equation is given below:

$$
\text{RMSD}(t) = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \left( r_i(t) - r_i^{\text{ref}} \right)^2 }
$$

### Purpose:
RMSD is used to evaluate the stability of the protein or molecule, check whether the system has reached equilibrium, and track significant structural changes over time.


**where:**


*   $r_i(t)-$  position of atom $i$ at time $t$
*   $r_i^{\text{ref}}$ → reference (initial) position of atom $i$
*   $N -$  total number of atoms



In [ ]:
#@title Calculate RMSD {run: "auto"}
!echo 3 3 | gmx rms -s "$pdb_filename" -f "$traj_filename" -o rmsd.xvg -tu ns



In [ ]:
# # If you want a more Pythonic way to call external programs,
# # you can uncomment and use this version!
# import subprocess
# try:
#     pdb_filename
#     traj_filename
# except NameError:
#     raise NameError("pdb_filename or traj_filename is undefined!")

# rmsdCmd = [
#     "gmx", "rms",
#     "-s", pdb_filename,
#     "-f", traj_filename,
#     "-o", "rmsd.xvg",
#     "-tu", "ns"
# ]

# input_text = "3\n3\n"

# process = subprocess.Popen(rmsdCmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
# stdout, stderr = process.communicate(input_text.encode())

# print(stdout.decode())
# print(stderr.decode())

# if process.returncode == 0:
#     print("RMSD analysis finished successfully.")
# else:
#     print("RMSD analysis failed!")

### Plot RMSD

This plot reads RMSD data from an XVG file and provides an interactive visualization of structural deviations of the protein over time. It helps to easily track the overall stability and conformational changes of the protein during the simulation.

In [ ]:
import numpy as np
import plotly.graph_objects as go

def read_xvg(filename):
  """Reads an XVG file using NumPy."""
  data = np.loadtxt(filename, comments=['@', '#'], dtype=float)
  return data

# Assuming your rmsd data is in a file named 'rmsd.xvg'
rmsd_data = read_xvg('rmsd.xvg')

# Extract x and y values from the data.  Adjust column indices if necessary.
time = rmsd_data[:, 0]
rmsd = rmsd_data[:, 1]


# Create an interactive plot using Plotly
fig = go.Figure(data=go.Scatter(x=time, y=rmsd, mode='lines+markers'))
fig.update_layout(
    title='RMSD Plot',
    xaxis_title='Time (ns)',  # Update x-axis label
    yaxis_title='RMSD (nm)',
    # xaxis=dict(range=[0.0, 0.5])
)

fig.show()


# Root Mean Square Fluctuation (RMSF)

### Description:
RMSF is a measure of how much each atom or residue deviates from its average position during a molecular dynamics simulation, providing information about flexibility. RMSF equation is given below:

$$
\text{RMSF}_i = \sqrt{\frac{1}{T} \sum_{t=1}^{T} \left( r_i(t) - \langle r_i \rangle \right)^2 }
$$

**where:**


*   $r_i(t)-$  position of atom $i$ at time $t$
*   $\langle r_i \rangle -$ average position of atom $i$ over the simulation
*   $T -$  total number of time frames


### Purpose:
The Root Mean Square Fluctuation (RMSF) is calculated to measure the mobility and flexibility of each atom within the protein structure over time. This metric highlights the fluctuations of atoms in different structural regions, providing insights into the dynamic behavior of the protein.

In [ ]:
 #@title Calculate RMSF {run: "auto"}
!echo 2 | gmx rmsf -s "{pdb_filename}" -f "{traj_filename}" -res -fit -oq rmsf.pdb


### Plot RMSF
**Reads RMSF** data from an **XVG** file and generates an interactive plot to visualize residue flexibility across the sequence.


In [ ]:
import numpy as np
import plotly.graph_objects as go

def read_xvg(filename):
  """Reads an XVG file using NumPy."""
  data = np.loadtxt(filename, comments=['@', '#'], dtype=float)
  return data

# Assuming your rmsf data is in a file named 'rmsf.xvg'
rmsf_data = read_xvg('rmsf.xvg')

# Extract x and y values from the data
residue_numbers = rmsf_data[:, 0]
rmsf_values = rmsf_data[:, 1]

# Create an interactive plot using Plotly
fig = go.Figure(data=go.Scatter(x=residue_numbers, y=rmsf_values, mode='lines+markers', marker_symbol='square'))
fig.update_layout(
    title='RMSF Plot',
    xaxis_title='Residue Number',
    yaxis_title='RMSF (nm)'
)

fig.show()


### Display 3D Structure Colored by RMSF (B-factor)

This code loads a PDB file (rmsf.pdb) and visualizes the protein in 3D using py3Dmol. It colors the protein according to B-factor values (corresponding to RMSF), using a rainbow gradient to highlight flexibility. The minimum and maximum B-factor values are automatically extracted from the file. The view is centered and zoomed, with labels and a legend showing the B-factor range. This allows identifying flexible and rigid regions and interpreting atomic mobility interactively.

In [ ]:
#@title Display 3D structure of protein colored accordding to RMSF values {run: "auto"}
import py3Dmol
import pandas as pd
import re

def get_bfactor_range(pdb_file):
    """
    Extract B-factor values from PDB file and return their range

    Parameters:
    pdb_file (str): Path to the PDB file

    Returns:
    tuple: (minimum B-factor, maximum B-factor)
    """
    bfactors = []

    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                try:
                    # B-factor is typically in columns 61-66
                    bfactor = float(line[60:66].strip())
                    bfactors.append(bfactor)
                except (ValueError, IndexError):
                    continue

    if not bfactors:
        return (0, 100)  # default range if no B-factors found

    return (min(bfactors), max(bfactors))

def visualize_protein_bfactor(pdb_file):
    """
    Visualize protein structure in cartoon representation colored by B-factor
    using automatically determined range and rainbow colors

    Parameters:
    pdb_file (str): Path to the PDB file
    """

    # Create a py3Dmol view instance
    view = py3Dmol.view()

    # Get B-factor range from the file
    bfactor_min, bfactor_max = get_bfactor_range(pdb_file)

    # Load the PDB file
    with open(pdb_file, 'r') as f:
        pdb_data = f.read()

    # Add the molecule to the viewer
    view.addModel(pdb_data, "pdb")

    # Set cartoon representation with rainbow coloring based on B-factor
    view.setStyle({'cartoon': {
        'colorscheme': {
            'prop': 'b',
            'gradient': 'linear',  # Using rainbow color scheme
            'min': bfactor_min,
            'max': bfactor_max,
            'colors': ["blue", "white", "red"]
        }
    }})

    # Center and zoom the view
    view.zoomTo()

    # Add legend for B-factor coloring
    view.addPropertyLabels(
        prop='b',
        gradient='bwr',
        min=bfactor_min,
        max=bfactor_max,
        legend={'x': 0.85, 'y': 0.5}
    )

    # Add text showing the B-factor range
    view.addLabel(f"B-factor range: {bfactor_min:.2f} - {bfactor_max:.2f}",
                 {'position': {'x': -20, 'y': -20, 'z': 0},
                  'backgroundColor': 'white',
                  'fontColor': 'black'})

    return view

# Replace with your PDB file path
pdb_file = "rmsf.pdb"
view = visualize_protein_bfactor(pdb_file)
view.show()

# Optional: Save the visualization as HTML
# view.save('protein_visualization.html')

# PCA Analysis

  Calculates the covariance matrix of atomic coordinates to perform Principal Component Analysis  (**PCA**) on protein motions. This helps to identify the dominant modes of motion, reduce dimensionality, and highlight collective structural changes in the protein during the simulation.

In [ ]:
#@title Calculate Covariance Matrix for PCA {run: "auto"}
!echo 3 3 | gmx covar -s "$pdb_filename" -f "$traj_filename" -o eigenval.xvg -v eigenvec.trr


### PCA Eigenvalues Plot (Interactive)

This code loads eigenvalues from the `eigenval.xvg `file and creates an interactive scatter plot using Plotly. The plot visualizes the variance captured by each principal component, helping to identify which components dominate the protein’s collective motions.

In [ ]:
import plotly.graph_objects as go

# Dosyayı yükle
eigenvec = np.loadtxt('eigenval.xvg', comments=['#', '@'])

# X ve Y bileşenlerini al
x = eigenvec[:, 0]
y = eigenvec[:, 1]

# Plotly ile interaktif grafik oluştur
fig = go.Figure(data=go.Scatter(x=x, y=y, mode='markers'))

fig.update_layout(
    title='Eigenvalues of the covariance matrix',
    xaxis_title='Eigenvector index',
    yaxis_title='$nm^2$'
)

fig.show()


### Perform PCA Projections

Runs GROMACS `anaeig` to project the trajectory onto the selected range of principal components, generating a 2D projection plot. This allows visualization of collective motions in the protein along the chosen principal components.

In [ ]:
for i in range(1, 4):
  for j in range(i+1, 4):
    firstComponent = i
    lastComponent = j
    !echo 3 3|gmx anaeig -s "$pdb_filename" -f "$traj_filename" \
      -first $firstComponent  -last $lastComponent  \
      -2d 2dproj_"$firstComponent"vs"$lastComponent".xvg \
      -extr -nframes 10

In [ ]:
playPrincipalComponent = "2" #@param ["1", "2", "3"]

In [ ]:
import py3Dmol

with open('extreme'+str(playPrincipalComponent)+'.pdb', 'r') as f:
    pdb = f.read()

v = py3Dmol.view(800, 600)
v.addModelsAsFrames(pdb)  # This is the key!
v.setStyle({'sphere': {}})

v.animate({'loop': 'forward', 'interval': 100})
v.zoomTo()
v  # Display it - will auto-animate

In [ ]:
#@title Select Principal Components to plot
firstComponent = "1" #@param ["1", "2", "3"]
lastComponent = "3" #@param ["1", "2", "3"]

In [ ]:
!echo $firstComponent
!echo $lastComponent

### 2D PCA Projection Plot

Reads the **2D PCA** projection file and creates an interactive scatter plot with color mapping to visualize the trajectory along the selected principal components. This allows tracking how the protein moves collectively in the plane defined by the chosen components.

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Read the data from the xvg file
data = np.loadtxt(f"2dproj_{firstComponent}vs{lastComponent}.xvg", comments=['@', '#'])

# Extract x, y values
x = data[:, 0]
y = data[:, 1]

# Create a color scale based on the order of the data points
colors = np.linspace(0, 1, len(x))  # Scale from 0 to 1

#e scatter plot with color mapping
fig = go.Figure(data=go.Scatter(
    x=x,
    y=y,
    mode='markers',
    marker=dict(
        color=colors,
        colorscale='RdBu',  # Use a red-blue color scale
        colorbar=dict(title="Data Point Order")
    )
))

fig.update_layout(
    title='2D Projection of PCA',
    xaxis_title=f'PC{firstComponent}',
    yaxis_title=f'PC{lastComponent}'
)

fig.show()


### K-Means Clustering on PCA Components

Performs K-Means clustering on the 2D PCA-projected data to group similar protein conformations together.

**K-Means:** An algorithm that partitions data into K clusters, minimizing the distance between points in the same cluster while maximizing the distance between clusters.

**Silhouette Score:** A metric used to determine the optimal number of clusters by measuring how well each point fits within its cluster.

Clustering is important because it helps identify patterns and dominant conformational states in the protein’s motion, revealing groups of similar collective movements along the selected principal components.

In [ ]:
from sklearn.cluster import KMeans
import plotly.express as px
import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Veri dosyasını yükle
data_2d = np.loadtxt(f"2dproj_{firstComponent}vs{lastComponent}.xvg", comments=['@', '#'])

# Seçilen PCA bileşenlerine göre veriyi al
X = data_2d[:, [0, 1]]

# Determine the optimal number of clusters using the Silhouette method
best_score = -1
best_k = 0

for k in range(2, 11):
  kmeans = KMeans(n_clusters=k, random_state=42).fit(X)
  score = silhouette_score(X, kmeans.labels_)
  if score > best_score:
    best_score = score
    best_k = k

print(f"Best number of clusters: {best_k}, Silhouette Score: {best_score}")


# K-Means kümeleme yap
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans.fit(X)
labels = kmeans.labels_


#Data plot with plotly.
#e scatter plot with color mapping
fig = go.Figure(data=go.Scatter(
    x=X[:, 0],
    y=X[:, 1],
    mode='markers',
    marker=dict(
        color=labels,
        colorscale='viridis',  # Use a red-blue color scale
        colorbar=dict(title="Data Point Order")
    )
))

fig.update_layout(
    title='K-Means Clustering on PCA Components PC{firstComponent} vs PC{lastComponent}',
    xaxis_title=f'PC{firstComponent}',
    yaxis_title=f'PC{lastComponent}'
)

fig.show()



# Solvent Accessible Surface Area (SASA) Calculation

Runs GROMACS SASA to calculate the solvent-accessible surface area of the selected group over the trajectory. This provides insights into the exposure of residues to the solvent, helping to understand protein folding, stability, and interactions with other molecules.

In [ ]:
!echo 1|gmx sasa -s "$pdb_filename" -f "$traj_filename" -o sasa.xvg -tu ns


### Plot SASA

Reads the SASA output file and creates an interactive plot showing the solvent-accessible surface area of the protein over time. This allows tracking how the protein’s surface exposure changes during the simulation and can indicate folding, unfolding, or conformational changes.

In [ ]:
import numpy as np
import plotly.graph_objects as go

def read_xvg(filename):
  """Reads an XVG file using NumPy."""
  data = np.loadtxt(filename, comments=['@', '#'], dtype=float)
  return data

# Assuming your rmsd data is in a file named 'rmsd.xvg'
rmsd_data = read_xvg('sasa.xvg')

# Extract x and y values from the data.  Adjust column indices if necessary.
time = rmsd_data[:, 0]
rmsd = rmsd_data[:, 1]


# Create an interactive plot using Plotly
fig = go.Figure(data=go.Scatter(x=time, y=rmsd, mode='lines+markers'))
fig.update_layout(
    title='SASA Plot',
    xaxis_title='Time (ns)',  # Update x-axis label
    yaxis_title= '$SASA (nm^2)$',
    # xaxis=dict(range=[0.0, 0.5])
)

fig.show()

# Radius of Gyration (Rg) Calculation

### Description:
The radius of gyration is a measure of the distribution of atoms around the protein’s center of mass, indicating how compact or extended the structure is during the simulation. Radius of Gyration (Rg) equation is given below:

$$
R_g = \sqrt{\frac{\sum_{i=1}^{N} m_i \, \| r_i - r_{com} \|^2}{\sum_{i=1}^{N} m_i}}
$$

**where:**  

- $m_i →$  mass of atom i  
- $r_i →$  position of atom i  
- $r_{\text{com}} →$ center of mass of the protein  
- $N →$ total number of atoms


In [ ]:
 #@title Calculate gyrate {run: "auto"}
!echo 3 3| gmx gyrate -s "$pdb_filename" -f "$traj_filename" -o gyrate.xvg

### Plot Radius of Gyration

Reads the gyration data file and creates an interactive plot showing the protein's radius of gyration over time. This allows monitoring the compactness of the protein and observing structural changes, folding, or unfolding events during the simulation.

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Read the gyrate.xvg file
data = np.loadtxt('gyrate.xvg', comments=['@', '#'])

# Extract time and gyration radius
time = data[:, 0]
gyration_radius = data[:, 1]

# Create the plot
fig = go.Figure(data=go.Scatter(x=time, y=gyration_radius, mode='lines+markers'))
fig.update_layout(
    title='Radius of Gyration',
    xaxis_title='Time (ps)',
    yaxis_title='Radius of Gyration (nm)'
)
fig.show()


# Secondary Structure Analysis

Protein secondary structure may not remain identical to the initial conformation during an MD simulation. The purpose of this analysis is to track secondary structure across the MD trajectory.

In [ ]:
#@title Install dssp and test it
!apt install dssp
!mkdssp --version


In [ ]:
#@title Calculate secondary structure
import MDAnalysis as mda
from MDAnalysis.analysis import dssp

u = mda.Universe(pdb_filename, traj_filename)
# Select protein residues from the Universe used for DSSP
protein = u.select_atoms("protein")

# Unique residues in correct order
residues = protein.residues

# Residue IDs exactly as in the PDB
res_ids = residues.resids

# Calculate secondary structure for each frame
dss = dssp.DSSP(u)

dss.run()

print(dss.results.dssp.shape)  # (n_frames, n_residues)

In [ ]:
#@title Define color parameters for secondary structures
import numpy as np

dssp_map = {
    "-": 0,   # Coil
    "H": 1,   # Alpha helix
    "G": 1,   # 3-10 helix
    "I": 1,   # Pi helix
    "E": 2,   # Beta sheet
    "B": 2,   # Beta bridge
    "T": 3,   # Turn
    "S": 3    # Bend
}

labels = {
    0: "Coil",
    1: "Helix",
    2: "Sheet",
    3: "Turn"
}

#ss_num = np.vectorize(dssp_map.get)(ss)
ss_num = np.vectorize(dssp_map.get)(dss.results.dssp)

#Define a discrete colorscale
colorscale = [
    [0.00, "#B3B3B3"], [0.25, "#B3B3B3"],  # Coil
    [0.25, "#D55E00"], [0.50, "#D55E00"],  # Helix
    [0.50, "#0072B2"], [0.75, "#0072B2"],  # Sheet
    [0.75, "#009E73"], [1.00, "#009E73"]   # Turn
]

In [ ]:
#@title Plot secondary structure
import plotly.graph_objects as go

fig = go.Figure(
    go.Heatmap(
        z=ss_num.T,
        x=np.arange(ss_num.shape[0]),   # frames or time
        y=res_ids,                      # REAL residue IDs
        colorscale=colorscale,
        zmin=0,
        zmax=3,
        colorbar=dict(
            tickmode="array",
            tickvals=[0, 1, 2, 3],
            ticktext=[labels[i] for i in range(4)],
            title="Secondary structure"
        )
    )
)
fig.data[0].customdata = dss.results.dssp.T
fig.data[0].hovertemplate = (
    "Residue ID: %{y}<br>"
    "Frame: %{x}<br>"
    "SS: %{customdata}<extra></extra>"
)

fig.update_layout(
    title="Secondary Structure Evolution",
    xaxis_title="Frame",
    yaxis_title="Residue ID (PDB numbering)",
    yaxis_autorange="reversed",   # DSSP-style top-to-bottom
    font=dict(size=14),
    height=650
)

fig.show()




# Cluster Analysis using GROMACS (gmx cluster)

### Purpose:

* Groups similar protein structures together, showing which conformations recur during the simulation.

* Identifies dominant conformational states of the protein.

* Helps analyze dynamic behavior and structural stability by examining the number, size, and composition of clusters.

* Simplifies visualization and interpretation by reducing many structures to representative clusters.

### What this code does:

* Uses gmx cluster to group similar protein conformations from a molecular dynamics trajectory.

* The analysis is based on RMSD (Root Mean Square Deviation) between frames.

* A cutoff value of 0.25 nm is used to decide whether two structures are considered part of the same cluster.

**Why it’s useful:**

* Helps identify recurring protein conformations along the trajectory.

* Reveals dominant structural states of the protein.

* Complements PCA/K-Means clustering by using full atomic coordinates instead of PCA-projected 2D data.

### Key points and differences from previous clustering (K-Means on PCA):

1. **Input data:**

* K-Means: PCA-reduced 2D vectors of protein motions.

* gmx cluster: Full atomic coordinates from the trajectory.

2. **Clustering method:**

* K-Means: Partitions data into a fixed number of clusters by minimizing distance to cluster centers.

* gmx cluster: Groups structures based on RMSD similarity, cutoff defines maximum allowed RMSD within a cluster.

3. **Cutoff (0.25 nm):**

* Two conformations are in the same cluster if their RMSD ≤ 0.25 nm.

* Smaller cutoff → more, tighter clusters.

* Larger cutoff → fewer, broader clusters.

4. **Purpose:**

* K-Means (PCA) highlights collective motions and main movement directions.

* gmx cluster identifies structurally similar states and dominant conformations along the trajectory

In [ ]:
#@title If there are large conformational changes, change this parameter to a bigger value!
rmsdCutoff = 0.3 #@param {type:"slider", min:0, max:1.5, step:0.05}
print(f"Selected RMSD cutoff is {rmsdCutoff}")

In [ ]:
#@title This process can take a long time depending on the rmsdCutoff you selected!
import subprocess

try:
    pdb_filename
    traj_filename
except NameError:
    raise NameError("pdb_filename or traj_filename is undefined!")

clusteringCmd = [
    "gmx", "cluster",
    "-s", pdb_filename,
    "-f", traj_filename,
    "-cutoff", str(rmsdCutoff),
    "-method", "gromos",
    "-o", "rmsd-matrix.xpm",
    "-g", "cluster.log",
    "-cl", "clusters.pdb"
]

input_text = "3\n1\n"

process = subprocess.Popen(clusteringCmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
stdout, stderr = process.communicate(input_text.encode())

print(stdout.decode())
print(stderr.decode())

if process.returncode == 0:
    print("Cluster analysis finished successfully.")
else:
    print("Cluster analysis failed!")


### Display Cluster Representative Structures (3D)

This code loads the cluster.pdb file containing representative protein structures from each cluster. Each cluster groups similar conformations observed along the trajectory. The 3D visualization colors each representative differently and labels the clusters, allowing easy identification of typical protein conformations and collective movements. This helps interpret which regions of the protein move together or remain stable

In [ ]:
import os
import py3Dmol

cluster_pdb_filename = "clusters.pdb"

if not os.path.exists(cluster_pdb_filename):
    print(f"'{cluster_pdb_filename}' not found!")
else:
    with open(cluster_pdb_filename, "r") as f:
        pdb_data = f.read()

    models = [m.strip() for m in pdb_data.split("ENDMDL") if m.strip()]

    view = py3Dmol.view(width=800, height=600)
    colors = ["red", "blue", "green", "orange", "purple",
              "cyan", "magenta", "yellow", "brown", "pink"]

    for i, model_data in enumerate(models):
        model_text = model_data
        if not model_text.endswith("ENDMDL"):
            model_text += "\nENDMDL"
        view.addModel(model_text, "pdb")
        view.setStyle({'model': i}, {"cartoon": {"color": colors[i % len(colors)]}})

    view.zoomTo()
    view.show()


In [ ]:
import os

logfile = "cluster.log"

if not os.path.exists(logfile):
    print("cluster.log not found! Make sure that you executed gmx cluster!")
else:
    with open(logfile, "r") as f:
        log_data = f.readlines()

    clusters = []
    for line in log_data:
        if line.strip().startswith("cl.") or "members" in line:
            clusters.append(line.strip())

    # Let's find the number of clusters.
    total = None
    for line in log_data:
        if "Found" in line and "clusters" in line:
            total = line.strip()

    if total:
        print("🧬 " + total + "\n")

    if clusters:
        print("List of clusters:\n")
        for cl in clusters:
            print(cl)
    else:
        print("No cluster found!")


In [ ]:
#@title Compress all generated data to analysis_results.tar.gz file.
analysis_files = [
    "rmsd.xvg",
    "rmsf.pdb",
    "rmsf.xvg",
    "eigenval.xvg",
    "eigenvec.trr",
    "covar.log",
    "2dproj_1vs2.xvg",
    "2dproj_1vs3.xvg",
    "2dproj_2vs3.xvg",
    "sasa.xvg",
    "gyrate.xvg",
    "rmsd-matrix.xpm",
    "cluster.log",
    "clusters.pdb",
    "average.pdb"
]

print("Created list of analysis files.")

In [ ]:
import tarfile
import os

archive_name = "analysis_results.tar.gz"

with tarfile.open(archive_name, "w:gz") as tar:
    for f_name in analysis_files:
        if os.path.exists(f_name):
            tar.add(f_name)
            print(f"Added {f_name} to {archive_name}")
        else:
            print(f"Warning: File {f_name} not found and will not be added to the archive.")

print(f"Successfully created {archive_name}")

**Congratulations! You can download all data in analysis_results.tar.gz file.**